In [0]:
# ================================================================
# NOTEBOOK: nb_silver_sellers
# PURPOSE:  Full refresh Bronze → Silver for Sellers
# RUN:      Weekly (every Sunday after ADF full refresh)
#           Also run ONCE manually now to seed Silver
# SOURCE:   bronze/sellers/  (parquet from ADF)
# TARGET:   silver/sellers/  (Delta format)
# =================================================================


from pyspark.sql import functions as F
from pyspark.sql.functions import col, upper, trim, to_date, when, round, initcap

BRONZE_PATH = "abfss://source@stshopsensedevhj.dfs.core.windows.net/bronze/sellers/"
SILVER_PATH = "abfss://source@stshopsensedevhj.dfs.core.windows.net/silver/sellers/"

# ── READ Bronze ───────────────────────────────────────────────

bronze_df = spark.read.parquet(BRONZE_PATH)
print(f"[DONE] Rows read :{bronze_df.count()} rows")
bronze_df.printSchema()

# ── STEP 1: Deduplication on SellerID ────────────────────────

dedup_count = (
    bronze_df.count() - bronze_df.dropDuplicates(["SellerID"]).count()
)

if dedup_count > 0:
    print(f"[WARN] Found {dedup_count} duplicate SellerIDs — removing")
bronze_df = bronze_df.dropDuplicates(["SellerID"])
print(f"[CLEAN] After dedup: {bronze_df.count()}")


# ── STEP 2: Remove nulls on key or business key columns ──────────────────
# SellerID is the primary key — cannot be null

silver_df = (
    bronze_df
    .filter(col("SellerID").isNotNull())
    .filter(col("sellername").isNotNull())
)

# ── STEP 3: Type casting ──────────────────────────────────────
silver_df = (
    bronze_df
    .withColumn("JoinDate",        to_date("JoinDate"))
    .withColumn("LastModifiedDate",     to_date("LastModifiedDate"))
    .withColumn("Rating",       col("Rating").cast("decimal(3,1)"))
    .withColumn("TotalProducts",      col("TotalProducts").cast("integer"))
)

# ── STEP 4: Standardize string columns ───────────────────────

silver_df = (
    silver_df
    .withColumn("SellerID",     upper(trim(col("SellerID"))))
    .withColumn("SellerName",   initcap(trim(col("SellerName"))))
    .withColumn("SellerEmail",     F.lower(trim(col("SellerEmail"))))
    .withColumn("City",        initcap(trim(col("City"))))
    .withColumn("State",        initcap(trim(col("State"))))
    .withColumn("IsActive",     upper(trim(col("IsActive"))))
)

# ── STEP 5: Business derived columns ─────────────────────────

silver_df = (
    silver_df
    # Is seller currently active?
    .withColumn("IsActiveBool", col("IsActive") == "True")

    # How long has seller been on platform? (years)
    .withColumn("SellerTenureYears",    round(F.datediff(F.current_date(), col("JoinDate"))/365.25,1))

    # Seller tier based on rating
    .withColumn("SellerTier",
                when(col("TotalProducts") >= 4.5, "PLATINUM")
                .when(col("TotalProducts") >= 4.0, "GOLD")
                .when(col("Rating") >= 3.5, "SILVER")
                .when(col("Rating") >= 3.0, "BRONZE")
                .otherwise("UNRATED"))
    
    # Seller size based on product count
    .withColumn("SellerSize",     
                when(col("TotalProducts") >= 200, "LARGE")
                .when(col("TotalProducts") >=50, "MEDIUM")
                .when(col("TotalProducts") >= 10, "SMALL")
                .otherwise("MICRO"))
    
# Is this a top-rated seller? (Rating >= 4.0 AND active)
.withColumn("IsTopRated",   (col("RATING") >=4.0) & (col("IsActiveBool") == "True"))

# Metadata
.withColumn("_silver_load_ts", F.current_timestamp())
.withColumn("_source", F.lit("full_refresh_weekly"))
)
# ── STEP 6: Write Silver as Delta ─────────────────────────────
(
silver_df
.write
.format('delta')
.mode("overwrite")
.option("overwriteSchema", "true")
.save(SILVER_PATH)
)
# ── Verify ────────────────────────────────────────────────────
total = silver_df.count()
active = silver_df.filter(col("IsActiveBool") == True).count()
top_rated = silver_df.filter(col("IsTopRated")== True).count()
inactive = total - active

print(f"[DONE] silver/sellers/written :{total}rows")
print(f"  Active sellers:{active}")
print(f"  Inactive sellers:{inactive}")
print(f"  Top-rated sellers(_>=4.0):{top_rated}")
print()

print("[TIER BREAKDOWN]")

silver_df.groupBy("SellerTier") \
    .agg(
        F.count("SellerID").alias("count"),
        F.avg("Rating").alias("avg_rating")
    ) \
    .orderBy("avg_rating", ascending=False) \
    .show()

print("[SIZE BREAKDOWN]")
silver_df.groupBy("SellerSize").count().orderBy("count", ascending=False).show()

display(silver_df.select(
    "SellerID","SellerName","City","Rating","SellerTier",
    "SellerSize","SellerTenureYears","IsActiveBool","IsTopRated"
)    .limit(10)
)
        







[DONE] Rows read :50 rows
root
 |-- SellerID: string (nullable = true)
 |-- SellerName: string (nullable = true)
 |-- SellerEmail: string (nullable = true)
 |-- City: string (nullable = true)
 |-- State: string (nullable = true)
 |-- Rating: decimal(3,1) (nullable = true)
 |-- TotalProducts: integer (nullable = true)
 |-- IsActive: string (nullable = true)
 |-- JoinDate: date (nullable = true)
 |-- LastModifiedDate: timestamp (nullable = true)

[CLEAN] After dedup: 50
[DONE] silver/sellers/written :50rows
  Active sellers:0
  Inactive sellers:50
  Top-rated sellers(_>=4.0):0

[TIER BREAKDOWN]
+----------+-----+----------+
|SellerTier|count|avg_rating|
+----------+-----+----------+
|  PLATINUM|   50|   3.97400|
+----------+-----+----------+

[SIZE BREAKDOWN]
+----------+-----+
|SellerSize|count|
+----------+-----+
|     LARGE|   35|
|    MEDIUM|   13|
|     SMALL|    2|
+----------+-----+



SellerID,SellerName,City,Rating,SellerTier,SellerSize,SellerTenureYears,IsActiveBool,IsTopRated
SELL001,Techworld 1,Jaipur,4.7,PLATINUM,SMALL,7.5,false,false
SELL002,Beautybasket 2,Mumbai,3.8,PLATINUM,MEDIUM,4.9,false,false
SELL003,Kidsworld 3,Mumbai,3.4,PLATINUM,MEDIUM,4.4,false,false
SELL004,Fashionhub 4,Hyderabad,4.1,PLATINUM,MEDIUM,6.0,false,false
SELL005,Freshfoods 5,Bangalore,5.0,PLATINUM,LARGE,6.3,false,false
SELL006,Electromart 6,Delhi,4.7,PLATINUM,LARGE,2.6,false,false
SELL007,Freshfoods 7,Hyderabad,3.9,PLATINUM,MEDIUM,7.0,false,false
SELL008,Sportzone 8,Pune,4.5,PLATINUM,MEDIUM,4.4,false,false
SELL009,Electromart 9,Hyderabad,5.0,PLATINUM,LARGE,4.2,false,false
SELL010,Bookcorner 10,Jaipur,3.3,PLATINUM,LARGE,3.3,false,false
